In [3]:
pip install pandas numpy scikit-learn joblib

In [6]:
import pandas as pd

df = pd.read_csv("cropdata_updated.csv")

print(df.head())
print(df.info())

  crop ID   soil_type Seedling Stage  MOI  temp  humidity  result
0   Wheat  Black Soil    Germination    1    25      80.0       1
1   Wheat  Black Soil    Germination    2    26      77.0       1
2   Wheat  Black Soil    Germination    3    27      74.0       1
3   Wheat  Black Soil    Germination    4    28      71.0       1
4   Wheat  Black Soil    Germination    5    29      68.0       1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16411 entries, 0 to 16410
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   crop ID         16411 non-null  object 
 1   soil_type       16411 non-null  object 
 2   Seedling Stage  16411 non-null  object 
 3   MOI             16411 non-null  int64  
 4   temp            16411 non-null  int64  
 5   humidity        16411 non-null  float64
 6   result          16411 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 897.6+ KB
None


In [8]:
from sklearn.preprocessing import LabelEncoder

# Create encoders
le_soil = LabelEncoder()
le_stage = LabelEncoder()
le_crop = LabelEncoder()

# Encode categorical columns
df['soil_type'] = le_soil.fit_transform(df['soil_type'])
df['Seedling Stage'] = le_stage.fit_transform(df['Seedling Stage'])
df['crop ID'] = le_crop.fit_transform(df['crop ID'])

In [10]:
X = df[['crop ID', 'soil_type', 'Seedling Stage', 'MOI', 'temp', 'humidity']]
y = df['result']

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=150, max_depth=10)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, n_estimators=150)

In [16]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9686262564727384
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1835
           1       0.95      1.00      0.97      1231
           2       0.94      0.64      0.76       217

    accuracy                           0.97      3283
   macro avg       0.96      0.87      0.91      3283
weighted avg       0.97      0.97      0.97      3283



In [18]:
import joblib

joblib.dump(model, "irrigation_model.pkl")
joblib.dump(le_soil, "soil_encoder.pkl")
joblib.dump(le_stage, "stage_encoder.pkl")
joblib.dump(le_crop, "crop_encoder.pkl")

['crop_encoder.pkl']

In [20]:
model = joblib.load("irrigation_model.pkl")
le_soil = joblib.load("soil_encoder.pkl")
le_stage = joblib.load("stage_encoder.pkl")
le_crop = joblib.load("crop_encoder.pkl")

In [22]:
def predict_irrigation(crop_id, soil, stage, moi, temp, humidity):
    
    # Encode inputs
    crop_id = le_crop.transform([crop_id])[0]
    soil = le_soil.transform([soil])[0]
    stage = le_stage.transform([stage])[0]

    data = [[crop_id, soil, stage, moi, temp, humidity]]

    prediction = model.predict(data)

    return prediction[0]

In [34]:
result = predict_irrigation(
    crop_id = "Wheat",
    soil = "Black Soil",
    stage = "Germination",
    moi=1,
    temp=25,
    humidity=80
)

print("Prediction:", result)

Prediction: 1


C:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import time

crop_id = "Wheat"
soil = "Black Soil"
stage = "Germination"


while True:
    # Replace with real sensor readings
    moi = 1  # soil moisture sensor
    temp = 25  # DHT11
    humidity = 80  # DHT11

    result = predict_irrigation(
        crop_id=crop_id,
        soil=soil,
        stage=stage,
        moi=moi,
        temp=temp,
        humidity=humidity
    )

    if result == 1:
        print("Irrigation Needed → Pump ON")
        # GPIO.output(pump_pin, GPIO.HIGH)
    else:
        print("No Irrigation → Pump OFF")
        # GPIO.output(pump_pin, GPIO.LOW)

    time.sleep(5)

In [ ]:
Serial.println("350,29,70");

In [ ]:
import serial

ser = serial.Serial('COM3', 9600)

while True:
    line = ser.readline().decode().strip()
    
    moi, temp, humidity = map(float, line.split(","))

    result = predict_irrigation(
        crop_id=CROP_ID,
        soil=SOIL_TYPE,
        stage=STAGE,
        moi=moi,
        temp=temp,
        humidity=humidity
    )

    print("Prediction:", result)

In [ ]:
import RPi.GPIO as GPIO

pump_pin = 18

GPIO.setmode(GPIO.BCM)
GPIO.setup(pump_pin, GPIO.OUT)

if result == 1:
    GPIO.output(pump_pin, GPIO.HIGH)
else:
    GPIO.output(pump_pin, GPIO.LOW)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")